# Đóng gói EcomGPT-7B từ ModelScope, đổi float32 .bin sang bfloat16 safetensors (Internet bật)

In [ ]:
import hashlib, json, shutil, subprocess, sys, time
from pathlib import Path
import requests
import torch
from safetensors.torch import save_file

MS = "https://www.modelscope.cn/api/v1/models/damo/nlp_ecomgpt_multilingual-7B-ecom/repo"
OUT = Path("/kaggle/working/EcomGPT-7B")
TMP = Path("/tmp/ecom_goc")
OUT.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)

ds = requests.get(MS + "/files", params={"Revision": "master", "Recursive": "true"}, timeout=120).json()["Data"]["Files"]
files = {f["Path"]: f for f in ds if f.get("Type") != "tree"}
shards = sorted(p for p in files if p.startswith("pytorch_model-") and p.endswith(".bin"))
print("shard goc:", [(p, round(files[p]["Size"] / 2**30, 2)) for p in shards], flush=True)
assert len(shards) == 4, shards

def tai(p, dich):
    for lan in range(4):
        try:
            with requests.get(MS, params={"Revision": "master", "FilePath": p}, stream=True, timeout=300) as r:
                r.raise_for_status()
                with open(dich, "wb") as f:
                    for c in r.iter_content(1 << 22):
                        f.write(c)
            if Path(dich).stat().st_size == files[p]["Size"]:
                return
            print("kich thuoc lech, tai lai", p, flush=True)
        except Exception as e:
            print("loi tai", p, lan, repr(e)[:200], flush=True)
            time.sleep(10)
    raise RuntimeError("khong tai duoc " + p)

for p in ["config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "README.md", "requirements.txt"]:
    tai(p, OUT / p)

weight_map, tong = {}, 0
for i, p in enumerate(shards, 1):
    t0 = time.time()
    goc = TMP / p
    tai(p, goc)
    print(f"[{i}/4] da tai {p} ({time.time() - t0:.0f} giay)", flush=True)
    sd = torch.load(goc, map_location="cpu", weights_only=True)
    goc.unlink()
    ra = {k: v.to(torch.bfloat16).contiguous() for k, v in sd.items()}
    del sd
    ten = f"model-{i:05d}-of-00004.safetensors"
    save_file(ra, str(OUT / ten), metadata={"format": "pt"})
    for k, v in ra.items():
        weight_map[k] = ten
        tong += v.numel() * 2
    print(f"[{i}/4] ghi {ten}: {len(ra)} tensor, {(OUT / ten).stat().st_size / 2**30:.2f} GiB", flush=True)
    del ra

(OUT / "model.safetensors.index.json").write_text(json.dumps({"metadata": {"total_size": tong}, "weight_map": weight_map}, indent=1), encoding="utf-8")
cfg = json.loads((OUT / "config.json").read_text(encoding="utf-8"))
cfg["torch_dtype"] = "bfloat16"
(OUT / "config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
sha = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.glob("*.safetensors"))}
(OUT / "REVISION.txt").write_text("damo/nlp_ecomgpt_multilingual-7B-ecom\nmaster (ModelScope), doi float32 .bin -> bfloat16 safetensors\n" + json.dumps(sha, indent=1) + "\n", encoding="utf-8")
tongdia = sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file())
print(f"TONG {tongdia / 2**30:.2f} GiB; so tensor {len(weight_map)}; tong tham so ~{tong / 2 / 1e9:.2f} ty", flush=True)
assert tongdia < 19 * 2**30, "vuot gioi han 20 GB"
# kiem nhanh: tokenizer va khung mo hinh (khong nap trong so)
import transformers
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
print("transformers", transformers.__version__, "torch", torch.__version__, flush=True)
tok = AutoTokenizer.from_pretrained(str(OUT))
for s in ["260GSM Boxy clean fit宽松短袖T恤洗水做旧矮胖男CROP CROPPED", "Tai nghe Bluetooth không dây kiểu kẹp tai thiết kế riêng thể thao"]:
    ids = tok(s)["input_ids"]
    print(len(s), "ky tu ->", len(ids), "token;", tok.decode(ids)[:60], flush=True)
c = AutoConfig.from_pretrained(str(OUT))
with torch.device("meta"):
    m = AutoModelForCausalLM.from_config(c)
print("khung mo hinh:", type(m).__name__, sum(p.numel() for p in m.parameters()) / 1e9, "ty tham so", flush=True)
print("XONG", flush=True)
